In [3]:
import pandas as pd

df = pd.read_csv("hf://datasets/agentlans/tatoeba-english-translations/All.csv.gz")

unique_sorted = sorted(df['Language'].unique())
print("Visos kalbos duomenyse:")
print(unique_sorted)


Visos kalbos duomenyse:
['\\N', 'abk', 'acm', 'ady', 'afb', 'afh', 'afr', 'aii', 'ain', 'ajp', 'akl', 'aln', 'alt', 'amh', 'ang', 'aoz', 'apc', 'ara', 'arg', 'arn', 'arq', 'ary', 'arz', 'asm', 'ast', 'ava', 'avk', 'awa', 'ayl', 'aze', 'bak', 'bal', 'bam', 'ban', 'bar', 'bcl', 'bel', 'ben', 'ber', 'bfz', 'bho', 'bis', 'bjn', 'bod', 'bom', 'bos', 'bre', 'brx', 'bua', 'bul', 'bvy', 'bzt', 'cat', 'cay', 'cbk', 'ceb', 'ces', 'cha', 'che', 'chg', 'chn', 'cho', 'chr', 'chv', 'cjy', 'ckb', 'ckt', 'cmn', 'cmo', 'cor', 'cos', 'cpi', 'crh', 'crk', 'crs', 'csb', 'cycl', 'cym', 'cyo', 'dan', 'deu', 'div', 'dng', 'drt', 'dsb', 'dtp', 'dws', 'egl', 'ell', 'emx', 'enm', 'epo', 'est', 'eus', 'ewe', 'ext', 'fao', 'fij', 'fin', 'fkv', 'fra', 'frm', 'frr', 'fry', 'fuc', 'fur', 'fuv', 'gaa', 'gag', 'gan', 'gbm', 'gcf', 'gil', 'gla', 'gle', 'glg', 'glv', 'gom', 'gos', 'got', 'grc', 'grn', 'gsw', 'guc', 'guj', 'guw', 'hak', 'hat', 'hau', 'haw', 'hax', 'hbo', 'hdn', 'heb', 'hif', 'hil', 'hin', 'hnj', 'hoc', '

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Kalbu grupė ( arba grupės)

kalbu_grupes = {
    "Germanų": ['eng', 'deu', 'nld', 'swe', 'afr', 'dan', 'nor', 'nds', 'yid', 'sco', 'fry', 'ltz', 'isl', 'fao'] }

# Taikyti modeliai

modeliai = {
    "LinearSVC": LinearSVC(),
    "LogReg": LogisticRegression(max_iter=2000, solver="saga", n_jobs=1),
    "NaiveBayes": MultinomialNB(),
}


#  Ciklas per visas kalbų grupės

for grupes_pavadinimas, kalbu_sarasas in kalbu_grupes.items():

    print("\n\n======================================")
    print(f"### KALBŲ GRUPĖ: {grupes_pavadinimas} ###")
    print("======================================\n")

    # 1. Filtruojame pasirinktas kalbas
    df_filtruotas = df[df['Language'].isin(kalbu_sarasas)]

    # 2. Randame mažiausią klasės dydį ir subalansuojame
    min_kiekis = df_filtruotas['Language'].value_counts().min()
    print("Mažiausias klasės dydis:", min_kiekis)

    df_balansuotas = pd.concat(
    [
        group.sample(min_kiekis, replace=False, random_state=123)
        for _, group in df_filtruotas.groupby("Language")
    ],
    ignore_index=True
    )

    print("\nSubalansuotos klasės:")
    print(df_balansuotas['Language'].value_counts())

    # 3. Pašaliname pasikartojančius tekstus
    df_balansuotas = df_balansuotas.drop_duplicates(subset=['Translation'])

    # 4. Paruošiame po=ymius
    X = df_balansuotas['Translation']
    y = df_balansuotas['Language']

    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5))
    X_vec = vectorizer.fit_transform(X)

    # 5. Padaliname į treniravimo ir testavimo rinkinius
    X_train, X_test, y_train, y_test = train_test_split(
        X_vec, y, test_size=0.2, random_state=123
    )

    # 6. Treniruojame modelius
    rezultatai = {}

    for modelio_pavad, modelis in modeliai.items():
        print(f"\n==== {modelio_pavad} ({grupes_pavadinimas}) ====")

        modelis.fit(X_train, y_train)
        y_pred = modelis.predict(X_test)

        tikslumas = accuracy_score(y_test, y_pred)
        rezultatai[modelio_pavad] = tikslumas

        print("Tikslumas:", tikslumas)
        print("\nKlasifikavimo ataskaita:")
        print(classification_report(y_test, y_pred))

    # 7. Grupės santrauka
    print("\n=== GRUPĖS SANTRAUKA:", grupes_pavadinimas, "===")
    for modelio_pavad, acc in rezultatai.items():
        print(f"{modelio_pavad}: {acc:.4f}")




### KALBŲ GRUPĖ: Germanų ###

Mažiausias klasės dydis: 94

Subalansuotos klasės:
Language
afr    94
dan    94
deu    94
fao    94
fry    94
isl    94
ltz    94
nds    94
nld    94
sco    94
swe    94
yid    94
Name: count, dtype: int64

==== LinearSVC (Germanų) ====
Tikslumas: 0.8878923766816144

Klasifikavimo ataskaita:
              precision    recall  f1-score   support

         afr       0.82      0.88      0.85        16
         dan       0.81      0.88      0.84        24
         deu       0.88      1.00      0.94        22
         fao       0.94      0.81      0.87        21
         fry       0.76      0.93      0.84        14
         isl       0.85      1.00      0.92        17
         ltz       1.00      0.84      0.91        19
         nds       0.84      0.89      0.86        18
         nld       0.85      0.79      0.81        14
         sco       1.00      0.88      0.94        17
         swe       0.94      0.77      0.85        22
         yid       1.00   

## Žemiau kodas tik su keturiom kalbom, kurias palaiko transformeris

In [8]:

# Kalbu grupė ( arba grupės)

kalbu_grupes = {
    "Germanų": ['deu', 'nld', 'swe', 'fry'] }

# Taikyti modeliai

modeliai = {
    "LinearSVC": LinearSVC(),
    "LogReg": LogisticRegression(max_iter=2000, solver="saga", n_jobs=1),
    "NaiveBayes": MultinomialNB(),
}


#  Ciklas per visas kalbų grupės

for grupes_pavadinimas, kalbu_sarasas in kalbu_grupes.items():

    print("\n\n======================================")
    print(f"### KALBŲ GRUPĖ: {grupes_pavadinimas} ###")
    print("======================================\n")

    # 1. Filtruojame pasirinktas kalbas
    df_filtruotas = df[df['Language'].isin(kalbu_sarasas)]

    # 2. Randame mažiausią klasės dydį ir subalansuojame
    min_kiekis = df_filtruotas['Language'].value_counts().min()
    print("Mažiausias klasės dydis:", min_kiekis)

    df_balansuotas = pd.concat(
    [
        group.sample(min_kiekis, replace=False, random_state=123)
        for _, group in df_filtruotas.groupby("Language")
    ],
    ignore_index=True
    )

    print("\nSubalansuotos klasės:")
    print(df_balansuotas['Language'].value_counts())

    # 3. Pašaliname pasikartojančius tekstus
    df_balansuotas = df_balansuotas.drop_duplicates(subset=['Translation'])

    # 4. Paruošiame po=ymius
    X = df_balansuotas['Translation']
    y = df_balansuotas['Language']

    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5))
    X_vec = vectorizer.fit_transform(X)

    # 5. Padaliname į treniravimo ir testavimo rinkinius
    X_train, X_test, y_train, y_test = train_test_split(
        X_vec, y, test_size=0.2, random_state=123
    )

    # 6. Treniruojame modelius
    rezultatai = {}

    for modelio_pavad, modelis in modeliai.items():
        print(f"\n==== {modelio_pavad} ({grupes_pavadinimas}) ====")

        modelis.fit(X_train, y_train)
        y_pred = modelis.predict(X_test)

        tikslumas = accuracy_score(y_test, y_pred)
        rezultatai[modelio_pavad] = tikslumas

        print("Tikslumas:", tikslumas)
        print("\nKlasifikavimo ataskaita:")
        print(classification_report(y_test, y_pred))

    # 7. Grupės santrauka
    print("\n=== GRUPĖS SANTRAUKA:", grupes_pavadinimas, "===")
    for modelio_pavad, acc in rezultatai.items():
        print(f"{modelio_pavad}: {acc:.4f}")





### KALBŲ GRUPĖ: Germanų ###

Mažiausias klasės dydis: 369

Subalansuotos klasės:
Language
deu    369
fry    369
nld    369
swe    369
Name: count, dtype: int64

==== LinearSVC (Germanų) ====
Tikslumas: 0.9790940766550522

Klasifikavimo ataskaita:
              precision    recall  f1-score   support

         deu       0.99      0.97      0.98        77
         fry       0.98      1.00      0.99        50
         nld       0.99      0.96      0.97        78
         swe       0.96      0.99      0.98        82

    accuracy                           0.98       287
   macro avg       0.98      0.98      0.98       287
weighted avg       0.98      0.98      0.98       287


==== LogReg (Germanų) ====
Tikslumas: 0.975609756097561

Klasifikavimo ataskaita:
              precision    recall  f1-score   support

         deu       0.99      0.97      0.98        77
         fry       0.94      1.00      0.97        50
         nld       0.99      0.95      0.97        78
         swe   